In [ ]:
# DISTILBERT-BASE-UNCASED — OPTIMAL PIPELINE (COLAB VERSION)

# CELL 1 — INSTALL & MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')

# Force clean uninstall of sklearn to prevent binary conflicts
!pip uninstall -y scikit-learn

# Install dependencies and UPGRADE PANDAS to match Kaggle's environment
!pip install -q -U pandas transformers>=4.48.0 accelerate>=1.2.0 evaluate datasets>=3.1.0 sentence-transformers bertopic umap-learn hdbscan==0.8.41 iterative-stratification rapidfuzz scikit-learn>=1.8.0

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - PLEASE ENABLE T4 GPU IN RUNTIME SETTINGS'}")
if torch.cuda.is_available():
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# CELL 2 — LOAD DATA
import pickle, numpy as np, os, time
from datetime import timedelta

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark   = True

# Paths for Colab
DATA = "/content/drive/MyDrive/nlp-genre-project-data/"
SAVE = "/content/"

with open(f"{DATA}data_splits.pkl", "rb") as f:
    splits = pickle.load(f)

X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]

with open(f"{DATA}mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
NUM_LABELS    = len(TARGET_GENRES)  # 12
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Labels: {NUM_LABELS} | Device: {device}")


# CELL 3 — UTILITIES
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, hamming_loss, classification_report

def fmt(seconds):
    return str(timedelta(seconds=int(seconds)))

def eta(elapsed_steps, total_steps, elapsed_seconds):
    if elapsed_steps == 0:
        return "calculating..."
    return fmt((elapsed_seconds / elapsed_steps) * (total_steps - elapsed_steps))

def get_pos_weight(y_train, device):
    counts = y_train.sum(axis=0)
    weight = (len(y_train) - counts) / (counts + 1e-8)
    return torch.FloatTensor(weight).to(device)

def tune_threshold_per_genre(probs, labels, num_labels, target_genres):
    """
    Per-genre threshold tuning- finds the optimal threshold
    independently for each genre rather than one global value.
    """
    best_thresholds = np.zeros(num_labels)
    print("  Per-genre threshold tuning:")
    for i in range(num_labels):
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.20, 0.85, 0.05):
            preds = (probs[:, i] > t).astype(int)
            f1    = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds[i] = best_t
        print(f"    {target_genres[i]:<20} best_t={best_t:.2f}  F1={best_f1:.4f}")
    return best_thresholds

def evaluate(model, loader, thresholds, model_name, device):
    """Full test-set evaluation using per-genre thresholds."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"], batch["attention_mask"])
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs > thresholds).astype(int)
            all_preds.append(preds)
            all_labels.append(batch["labels"].cpu().numpy())
    P, L = np.vstack(all_preds), np.vstack(all_labels)
    print(f"\n=== {model_name} Test Results (per-genre thresholds) ===")
    print(f"Micro-F1:     {f1_score(L, P, average='micro', zero_division=0):.4f}")
    print(f"Macro-F1:     {f1_score(L, P, average='macro', zero_division=0):.4f}")
    print(f"Hamming Loss: {hamming_loss(L, P):.4f}")
    print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))


# CELL 4 — VRAM-CACHED DATASET + MODEL
MODEL_NAME = "distilbert-base-uncased"

class UltraFastGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512, device="cuda"):
        self.labels = torch.FloatTensor(labels).to(device)
        print(f"Tokenizing {len(texts)} texts → VRAM...")
        encodings = tokenizer(
            list(texts),
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.input_ids      = encodings["input_ids"].to(device)
        self.attention_mask = encodings["attention_mask"].to(device)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx]
        }

class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(0.3)
        self.clf  = nn.Linear(768, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.clf(self.drop(cls))


# CELL 5 — LOAD INTO VRAM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tok_start = time.time()
print("\n--- Loading data directly into GPU VRAM ---")
train_ds = UltraFastGenreDataset(X_train, y_train, tokenizer, device=device)
val_ds   = UltraFastGenreDataset(X_val,   y_val,   tokenizer, device=device)
test_ds  = UltraFastGenreDataset(X_test,  y_test,  tokenizer, device=device)
print(f"All splits loaded in {fmt(time.time() - tok_start)}")

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM after data load: {used:.1f} GB / {total:.1f} GB ({100*used/total:.1f}% used)")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


# CELL 6 — TRAINING
model     = DistilBERTClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS      = 8
total_steps = len(train_loader) * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_macro            = 0.0
best_thresholds_saved = np.full(NUM_LABELS, 0.5)
patience              = 5
no_improve            = 0
train_start           = time.time()
global_step           = 0

print("\n" + "=" * 65)
print(f"DistilBERT optimal training started.")
print(f"Total steps: {total_steps} | Steps/epoch: {len(train_loader)}")
print(f"Batch size: 32 | Max len: 512 | Precision: FP16")
print(f"Epochs: {EPOCHS} | Patience: {patience} | Scheduler: Cosine")
print(f"Threshold: per-genre sweep 0.20 → 0.85")
print(f"Estimated total time: ~2-3 hours")
print("=" * 65)

for epoch in range(EPOCHS):
    epoch_start  = time.time()
    running_loss = 0.0
    model.train()

    print(f"\n{'─'*65}")
    print(f"EPOCH {epoch+1}/{EPOCHS}  |  Elapsed: {fmt(time.time() - train_start)}")
    print(f"{'─'*65}")

    for step, batch in enumerate(train_loader):
        step_start = time.time()
        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss   = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        global_step  += 1
        step_time     = time.time() - step_start

        if (step + 1) % 100 == 0:
            avg_loss       = running_loss / (step + 1)
            epoch_elapsed  = time.time() - epoch_start
            total_elapsed  = time.time() - train_start
            steps_left_ep  = len(train_loader) - (step + 1)
            epoch_eta_secs = (epoch_elapsed / (step + 1)) * steps_left_ep

            print(
                f"  Step {step+1:>4}/{len(train_loader)}"
                f"  |  Loss: {avg_loss:.4f}"
                f"  |  Step: {step_time:.2f}s"
                f"  |  Epoch ETA: {fmt(epoch_eta_secs)}"
                f"  |  Total ETA: {eta(global_step, total_steps, total_elapsed)}"
            )

    epoch_duration = time.time() - epoch_start
    print(f"\nEpoch {epoch+1} training done in {fmt(epoch_duration)}")

    # Validation
    val_start = time.time()
    model.eval()
    all_logits, all_labels_val = [], []

    with torch.no_grad():
        for b in val_loader:
            logits = model(b["input_ids"], b["attention_mask"])
            all_logits.append(logits.cpu().numpy())
            all_labels_val.append(b["labels"].cpu().numpy())

    val_duration  = time.time() - val_start
    all_logits_np = np.vstack(all_logits)
    all_labels_np = np.vstack(all_labels_val)
    all_probs_np  = 1 / (1 + np.exp(-all_logits_np))

    best_t_epoch = tune_threshold_per_genre(all_probs_np, all_labels_np, NUM_LABELS, TARGET_GENRES)
    best_preds_epoch = (all_probs_np > best_t_epoch).astype(int)
    ep_macro_f1      = f1_score(all_labels_np, best_preds_epoch, average="macro", zero_division=0)

    total_elapsed = time.time() - train_start
    print(f"\nValidation done in {fmt(val_duration)}")
    print(f"Epoch {epoch+1}/{EPOCHS}  |  Val Macro-F1: {ep_macro_f1:.4f}")
    print(f"Total elapsed: {fmt(total_elapsed)}  |  Remaining ETA: {eta(global_step, total_steps, total_elapsed)}")

    if ep_macro_f1 > best_macro:
        best_macro            = ep_macro_f1
        best_thresholds_saved = best_t_epoch.copy()
        no_improve            = 0
        torch.save(model.state_dict(), f"{SAVE}distilbert_optimal_best.pt")
        print("  -> Saved best model")
        print(f"  -> Thresholds: {dict(zip(TARGET_GENRES, best_t_epoch.round(2)))}")
    else:
        no_improve += 1
        print(f"  -> No improvement ({no_improve}/{patience})")

    if no_improve >= patience:
        print("Early stopping triggered.")
        break

total_time = time.time() - train_start
print(f"\n{'='*65}")
print(f"Training complete. Total time: {fmt(total_time)}")
print(f"Best Val Macro-F1: {best_macro:.4f}")
print(f"{'='*65}")


# CELL 7 — FINAL EVALUATION
eval_start = time.time()

print("\nLoading best model for final test evaluation...")
model.load_state_dict(torch.load(f"{SAVE}distilbert_optimal_best.pt", map_location=device))

model.eval()
all_logits_val, all_labels_val = [], []
with torch.no_grad():
    for b in val_loader:
        logits = model(b["input_ids"], b["attention_mask"])
        all_logits_val.append(logits.cpu().numpy())
        all_labels_val.append(b["labels"].cpu().numpy())

all_probs_val  = 1 / (1 + np.exp(-np.vstack(all_logits_val)))
all_labels_val = np.vstack(all_labels_val)

final_thresholds = tune_threshold_per_genre(all_probs_val, all_labels_val, NUM_LABELS, TARGET_GENRES)

evaluate(model, test_loader, final_thresholds, "DistilBERT-Optimal", device)

print(f"\nEvaluation done in {fmt(time.time() - eval_start)}")
print(f"Total session time: {fmt(time.time() - train_start)}")


# CELL 8 — BACKUP MODEL TO DRIVE
import shutil
import os

source_model = f"{SAVE}distilbert_optimal_best.pt"
drive_backup = f"{DATA}distilbert_optimal_best.pt"

print("\nStarting model backup...")

if os.path.exists(source_model):
    shutil.copy2(source_model, drive_backup)
    print(f" Success! Model securely backed up to: {drive_backup}")
else:
    print(f" Error: Could not find the model at {source_model}. Did the training complete and save successfully?")

Mounted at /content/drive
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.

PyTorch: 2.10.0+cu128
GPU:     Tesla T4
VRAM:    15.6 GB
Train: 36104, Val: 7774, Test: 7740
Labels: 12

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


--- Loading data directly into GPU VRAM ---
Tokenizing 36104 texts → VRAM...
Tokenizing 7774 texts → VRAM...
Tokenizing 7740 texts → VRAM...
All splits loaded in 0:00:55
VRAM after data load: 0.4 GB / 15.6 GB (2.7% used)
Train batches: 1129 | Val: 243 | Test: 242


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



DistilBERT optimal training started.
Total steps: 9032 | Steps/epoch: 1129
Batch size: 32 | Max len: 512 | Precision: FP16
Epochs: 8 | Patience: 5 | Scheduler: Cosine
Threshold: per-genre sweep 0.20 → 0.85
Estimated total time: ~2-3 hours

─────────────────────────────────────────────────────────────────
EPOCH 1/8  |  Elapsed: 0:00:00
─────────────────────────────────────────────────────────────────
  Step  100/1129  |  Loss: 1.2320  |  Step: 0.36s  |  Epoch ETA: 0:06:14  |  Total ETA: 0:54:13
  Step  200/1129  |  Loss: 1.1707  |  Step: 0.38s  |  Epoch ETA: 0:05:40  |  Total ETA: 0:53:54
  Step  300/1129  |  Loss: 1.1126  |  Step: 0.37s  |  Epoch ETA: 0:05:07  |  Total ETA: 0:53:58
  Step  400/1129  |  Loss: 1.0419  |  Step: 0.37s  |  Epoch ETA: 0:04:30  |  Total ETA: 0:53:21
  Step  500/1129  |  Loss: 0.9856  |  Step: 0.37s  |  Epoch ETA: 0:03:53  |  Total ETA: 0:52:52
  Step  600/1129  |  Loss: 0.9446  |  Step: 0.37s  |  Epoch ETA: 0:03:16  |  Total ETA: 0:52:19
  Step  700/1129  | 